# Lab 10 -- Decision Trees

Compared to last week, this is a very simple lab <span style="font-size:20pt;">😃</span> You'll have fun programming!

You will implement the **Classification and Regression Tree (CART)** algorithm from scratch.

The lab is broken down into the following pieces:

* Regression Criterion
* Creating Splits
* Buiding a Tree
* Making a prediction


# Decision trees for Regression
## Exercise 1 -- Download and load the dataset

We will be using the usual Boston Housing dataset, which is available to download from ECLASS

* Download the file
* Read it and separate the target variable from the features.
* Make a 80/10/10 train/validation/test split

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
housing_names = ["CRIM", "ZN", "INDUS", "CHAS", "NOX", "RM", "AGE", "DIS", "RAD", "TAX", "PTRATIO", "B", "LSTAT", "MEDV"]
data = pd.read_table("housing.txt", names=housing_names, sep="\s+")

<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
D:\Users\C3007807\AppData\Local\Temp\ipykernel_23796\2240041893.py:2: SyntaxWarning: invalid escape sequence '\s'
  data = pd.read_table("housing.txt", names=housing_names, sep="\s+")


The target variable will be as usual `MEDV`. Use the rest as features.

In [3]:
X = data.iloc[:, :-1].values
y = data["MEDV"].values

In [4]:
X.shape

(506, 13)

In [5]:
X_train, X_aux, y_train, y_aux = train_test_split(X, y, train_size= 0.8)
X_test, X_val, y_test, y_val = train_test_split(X_aux, y_aux, train_size= 0.5)


## Exercise 2 -- Optimization Criterion

For regression, a simple criterion to optimize is to minimize the sum of squared errors for a given region. This is, for all datapoints in a region with size, we minimize:

$$\sum_{i=1}^N(y_i - \hat{y})^2$$

where $N$ is the number of datapoits in the region and $\hat{y}$ is the mean value of the region for the target variable. 

Implement such a function using the description below.

Please, don't use an existing implementation, refer to the [book](https://www.statlearning.com/s/ISLRSeventhPrinting.pdf), and if you need help, ask questions!

In [6]:
def regression_criterion(region: np.ndarray):
    """
    Implements the sum of squared error criterion in a region
    
    Parameters
    ----------
    region : ndarray
        Array of shape (N,) containing the values of the target values 
        for N datapoints in the training set.
    
    Returns
    -------
    float
        The sum of squared error
        
    Note
    ----
    The error for an empty region should be infinity (use: float("inf"))
    This avoids creating empty regions
    """
    
    if region.size > 0: mean = region.mean()
    else: return float("inf")
    
    return np.sum((region - mean)**2)

In [7]:
# test your code
rng = np.random.default_rng(0)
print(regression_criterion(rng.random(size=40)))
print(regression_criterion(np.ones(10)))
print(regression_criterion(np.zeros(10)))
print(regression_criterion(np.array([])))

3.6200679838629544
0.0
0.0
inf


## Exercise 3 -- Make a split

In [8]:
def split_region(region, feature_index, tau):
    """
    Given a region, splits it based on the feature indicated by
    `feature_index`, the region will be split in two, where
    one side will contain all points with the feature with values 
    lower than `tau`, and the other split will contain the 
    remaining datapoints.
    
    Parameters
    ----------
    region : array of size (n_samples, n_features)
        a partition of the dataset (or the full dataset) to be split
    feature_index : int
        the index of the feature (column of the region array) used to make this partition
    tau : float
        The threshold used to make this partition
        
    Return
    ------
    left_partition : array
        indices of the datapoints in `region` where feature < `tau`
    right_partition : array
        indices of the datapoints in `region` where feature >= `tau` 
    """
    
    left_partition = region[:, feature_index] < tau
    right_partition = region[:, feature_index] >= tau
    
    return left_partition, right_partition

## Exercise 4 -- Find the best split

The strategy is quite simple (as well as inefficient), but it helps to reinforce the concepts.
We are going to use a greedy, exhaustive algorithm to select splits, selecting the `feature_index` and the `tau` that minimizes the Regression Criterion

In [9]:
def get_split(X, y):
    """
    Given a dataset (full or partial), splits it on the feature of that minimizes the sum of squared error
    
    Parameters
    ----------
    X : array (n_samples, n_features)
        features 
    y : array (n_samples, )
        labels
    
    Returns
    -------
    decision : dictionary
        keys are:
        * 'feature_index' -> an integer that indicates the feature (column) of `X` on which the data is split
        * 'tau' -> the threshold used to make the split
        * 'left_region' -> array of indices where the `feature_index`th feature of X is lower than `tau`
        * 'right_region' -> indices not in `low_region`
    """
    n_samples, n_features = X.shape
    
    best_sse = float("inf")
    best_feature = 0
    best_tau = 0
    best_left_indices = 0
    best_right_indices = 0
    
    for feature_idx in range(n_features):
        thresholds = X[:, feature_idx]

        for tau in thresholds:
            # left_indices = np.where(X[:, feature_idx] < tau)[0]
            # right_indices = np.where(X[:, feature_idx] >= tau)[0]
            left_indices, right_indices = split_region(X, feature_idx, tau)
        
            current_sse = regression_criterion(y[left_indices]) + regression_criterion(y[right_indices])
            
            if current_sse < best_sse:
                best_sse = current_sse
                best_feature = feature_idx
                best_tau = tau
                best_left_indices = left_indices
                best_right_indices = right_indices
                
    return {'feature_index': int(best_feature),
            'tau': float(best_tau),
            'left_region': best_left_indices,
            'right_region': best_right_indices}
    

## Exercise 5 -- Recursive Splitting

The test above is an example on how to find the root node of our decision tree. The algorithm now is a greedy search until we reach a stop criterion. To find the actual root node of our decision tree, you must provide the whole training set, not just a slice of 15 rows as the test above.

The trivial stopping criterion is to recursively grow the tree until each split contains a single point (perfect node purity). If we go that far, it normally means we are overfitting.

You will implement these criteria to stop the growth:

* A node is a leaf if:
    * It has less than `min_samples` datapoints
    * It is at the `max_depth` level from the root (each split creates a new level)
    * The criterion is `0`



In [10]:
root = get_split(X_train, y_train)

In [11]:
def recursive_growth(node, min_samples, max_depth, current_depth, X, y):
    """
    Recursively grows a decision tree.
    
    Parameters
    ----------
    node : dictionary
        If the node is terminal, it contains only the "value" key, which determines the value to be used as a prediction.
        If the node is not terminal, the dictionary has the structure defined by `get_split`
    min_samples : int
        parameter for stopping criterion if a node has <= min_samples datapoints
    max_depth : int
        parameter for stopping criterion if a node belongs to this depth
    depth : int
        current distance from the root
    X : array (n_samples, n_features)
        features (full dataset)
    y : array (n_samples, )
        labels (full dataset)
    
    Notes
    -----
    To create a terminal node, a dictionary is created with a single "value" key, with a value that
    is the mean of the target variable
    
    'left' and 'right' keys are added to non-terminal nodes, which contain (possibly terminal) nodes 
    from higher levels of the tree:
    'left' corresponds to the 'left_region' key, and 'right' to the 'right_region' key
    """
    
    left_indices = node["left_region"]
    right_indices = node["right_region"]
    
    X_left, y_left = X[left_indices], y[left_indices]
    X_right, y_right = X[right_indices], y[right_indices]
    
    if (len(y_left) < min_samples or current_depth >= max_depth or regression_criterion(y_left) == 0):
        node["left_region"] = {"value": y_left.mean() if len(y_left) > 0 else 0.0}
    else:
        node["left_region"] = get_split(X_left, y_left)
        recursive_growth(node["left_region"], min_samples, max_depth, current_depth + 1, X_left, y_left)
    
    if (len(y_right) < min_samples or current_depth >= max_depth or regression_criterion(y_right) == 0):
        node["right_region"] = {"value": y_right.mean() if len(y_right) > 0 else 0.0}
    else:
        node["right_region"] = get_split(X_right, y_right)
        recursive_growth(node["right_region"], min_samples, max_depth, current_depth + 1, X_right, y_right)
        

In [12]:
recursive_growth(root, 20, 2, 0, X_train, y_train)

In [13]:
a = np.array([0,2,3,4,5,6,7,8,9])
b = np.array([0,1])
a[b]

array([0, 2])

Below we provide code to visualise the generated tree!

In [14]:
def print_tree(node, depth=0):
    """Recursively prints a human-readable representation of the tree."""
    indent = ' . ' * depth
    if 'value' in node:
        print(f"{indent}predict = {node['value']:.4f}")
    else:
        print(f"{indent}X[{node['feature_index']}] < {node['tau']:.4f}")
        print(f"{indent}  left:")
        print_tree(node['left_region'],  depth + 1)
        print(f"{indent}  right:")
        print_tree(node['right_region'], depth + 1)



In [15]:
print_tree(root, 0)

X[12] < 9.6400
  left:
 . X[5] < 7.4540
 .   left:
 .  . X[7] < 1.6132
 .  .   left:
 .  .  . predict = 50.0000
 .  .   right:
 .  .  . predict = 26.5420
 .   right:
 .  . X[0] < 3.4743
 .  .   left:
 .  .  . predict = 46.2083
 .  .   right:
 .  .  . predict = 21.9000
  right:
 . X[12] < 15.0200
 .   left:
 .  . X[5] < 6.8790
 .  .   left:
 .  .  . predict = 20.5160
 .  .   right:
 .  .  . predict = 26.7833
 .   right:
 .  . X[0] < 6.3931
 .  .   left:
 .  .  . predict = 16.8015
 .  .   right:
 .  .  . predict = 11.9833


## Exercise 6 -- Make a Prediction
Use the a node to predict the class of a compatible dataset

In [16]:
def predict_sample(node, sample):
    """
    Makes a prediction based on the decision tree defined by `node`
    
    Parameters
    ----------
    node : dictionary
        A node created one of the methods above
    sample : array of size (n_features,)
        a sample datapoint
    """
    if "value" in node:
        return node["value"]
    if sample[node["feature_index"]] < node["tau"]:
        return predict_sample(node["left_region"], sample)
    else:
        return predict_sample(node["right_region"], sample)
    
        
def predict(node, X):
    """
    Makes a prediction based on the decision tree defined by `node`
    
    Parameters
    ----------
    node : dictionary
        A node created one of the methods above
    X : array of size (n_samples, n_features)
        n_samples predictions will be made
    """
    return np.array([predict_sample(node, sample) for sample in X])

Now use the functions defined above to calculate the RMSE of the validation set. 
* Try first with `min_samples=20` and `max_depth=6` (for this values you should get a validation RMSE of ~8.8)

Then, experiment with different values for the stopping criteria.

In [17]:
# calculate root mean squared error with numpy
def root_mean_squared_error(y_true, y_pred):
    """
    Calculates the root mean squared error between two arrays
    
    Parameters
    ----------
    y_true : array of size (n_samples,)
        true labels
    y_pred : array of size (n_samples,)
        predicted labels
    """
    return np.sqrt(np.mean((y_true - y_pred)**2))

In [18]:
root = get_split(X_train, y_train)
min_samples = 20
max_depth = 6
recursive_growth(root, min_samples, max_depth, 1, X_train, y_train)
train_mse = root_mean_squared_error(y_train, predict(root, X_train))
test_mse = root_mean_squared_error(y_test, predict(root, X_test))

print(f'Train MSE : {train_mse}')
print(f'Test MSE : {test_mse}')

Train MSE : 2.5682391995810936
Test MSE : 3.2030258164326963


## Just to see how much we can improve this naive decision tree!

In [19]:
from sklearn.linear_model import LinearRegression

In [20]:
reg = LinearRegression().fit(X_train, y_train)

In [21]:
reg_train_mse = root_mean_squared_error(y_train, reg.predict(X_train))
reg_test_mse = root_mean_squared_error(y_test, reg.predict(X_test))
print(f'Regression Train MSE : {reg_train_mse}')
print(f'Regression Test MSE : {reg_test_mse}')

Regression Train MSE : 4.774947051145968
Regression Test MSE : 4.4714878070831725


# Decision trees for Classification
You will implement decision trees for classification using the Gini index as the splitting criterion. You’ll build the tree recursively, selecting splits that minimize Gini impurity and classifying samples based on majority class in each leaf. A good dataset to start with is the Iris dataset, which is small, well-labeled, and available directly via `sklearn.datasets.load_iris().`

In [22]:
import sklearn

In [23]:
# Load the Iris dataset
data = sklearn.datasets.load_iris()
X = data.data
y = data.target

We will focus only on binary classification today!

In [24]:
X = X[y != 2]
y = y[y != 2]

In [25]:
X_train, X_aux, y_train, y_aux = train_test_split(X, y, train_size=0.8, random_state=42)
X_test, X_val, y_test, y_val = train_test_split(X_aux, y_aux, train_size=0.5, random_state=42)

### Feel free to use the same code as for regression, but change the criterion and the prediction function. You can also implement a new one if you want to!

In [26]:
def gini_criterion(region):
    if region.size == 0:
        return float("inf")
    classes = np.unique(region)
    gini = 1.0
    for c in classes:
        p = np.sum(region == c) / region.size
        gini -= p ** 2
    return gini * region.size  


def get_split(X, y):
    n_samples, n_features = X.shape
    best_gini = float("inf")
    best_feature, best_tau = 0, 0
    best_left, best_right = None, None

    for feature_idx in range(n_features):
        for tau in np.unique(X[:, feature_idx]):
            left  = np.where(X[:, feature_idx] < tau)[0]
            right = np.where(X[:, feature_idx] >= tau)[0]
            score = gini_criterion(y[left]) + gini_criterion(y[right])
            if score < best_gini:
                best_gini, best_feature, best_tau = score, feature_idx, tau
                best_left, best_right = left, right

    return {"feature_index": int(best_feature), "tau": float(best_tau),
            "left_region": best_left, "right_region": best_right}
    
def leaf_val(labels):
        if len(labels) == 0:
            return 0
        return int(np.bincount(labels).argmax())

def recursive_growth(node, min_samples, max_depth, current_depth, X, y):
    left_idx, right_idx = node["left_region"], node["right_region"]
    X_l, y_l = X[left_idx], y[left_idx]
    X_r, y_r = X[right_idx], y[right_idx]

    
    if (len(y_l) == 0 or len(y_l) <= min_samples or current_depth >= max_depth or len(np.unique(y_l)) == 1):
        node["left_region"] = {"value": leaf_val(y_l)}
    else:
        node["left_region"] = get_split(X_l, y_l)
        recursive_growth(node["left_region"], min_samples, max_depth, current_depth + 1, X_l, y_l)

    if (len(y_r) == 0 or len(y_r) <= min_samples or current_depth >= max_depth or len(np.unique(y_r)) == 1):
        node["right_region"] = {"value": leaf_val(y_r)}
    else:
        node["right_region"] = get_split(X_r, y_r)
        recursive_growth(node["right_region"], min_samples, max_depth, current_depth + 1, X_r, y_r)


def predict_sample(node, sample):
    if "value" in node:
        return node["value"]
    if sample[node["feature_index"]] < node["tau"]:
        return predict_sample(node["left_region"], sample)
    return predict_sample(node["right_region"], sample)

def predict(node, X):
    return np.array([predict_sample(node, s) for s in X])


In [27]:
def accuracy(y_true, y_pred): return np.mean(y_true == y_pred)

best_acc, best_min_samples, best_root = -1, None, None

for min_s in [2, 4, 6]:
    root = get_split(X_train, y_train)
    recursive_growth(root, min_samples=min_s, max_depth=10,
                         current_depth=1, X=X_train, y=y_train)
    
    val_acc = accuracy(y_val, predict(root, X_val))
    print(f"min_samples={min_s}  ->  val accuracy = {val_acc:.4f}")
    
    if val_acc > best_acc:
        best_acc, best_min_samples, best_root = val_acc, min_s, root

print(f"\nBest min_samples = {best_min_samples}  (val acc = {best_acc:.4f})")


min_samples=2  ->  val accuracy = 1.0000
min_samples=4  ->  val accuracy = 1.0000
min_samples=6  ->  val accuracy = 1.0000

Best min_samples = 2  (val acc = 1.0000)


Use accuracy to find the best split. Don't import it from sklearn, calculate it yourself, it's a one-liner ;)

In [28]:
test_acc = accuracy(y_test, predict(best_root, X_test))
print(f'Test accuracy (min_samples={best_min_samples}): {test_acc:.4f}')

Test accuracy (min_samples=2): 1.0000
